# Interactive SpaceX dashboard

**AI-assisted research coursework for Amir Saulebekov.** The accompanying spacex_dashboard.py runs a real Dash app. Chrome verification confirms the site dropdown and payload range update the pie and scatter charts. This notebook executes the callback checks without starting a blocking web server.

In [1]:
import sys
from pathlib import Path
import json
sys.path.insert(0,str(Path('..').resolve()))
import spacex_dashboard as dashboard
print('Source flights:',len(dashboard.df))

Source flights: 90


In [2]:
print(dashboard.app.server.test_client().get('/').status_code)

200


The script provides the requested all-site success-count pie, an individual-site success/failure pie, and payload/outcome scatter colored by booster category. Inspect the source below for the exact callback function and data path. Launch locally with `python spacex_dashboard.py` and open http://127.0.0.1:8055/. The source discloses the historical dates and unmatched SQL booster category.

In [3]:
print(Path('../spacex_dashboard.py').read_text())

"""Historical course dashboard; run locally on 127.0.0.1:8055."""
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash,dcc,html,Input,Output
DATA=Path(__file__).resolve().parent/'dashboard_launches.csv'
df=pd.read_csv(DATA)
app=Dash(__name__)
app.layout=html.Main([
 html.H1('SpaceX launches: a historical course dataset'),
 html.P('90 recorded flights, 2010–2020. Success follows the course landing label and includes successful ocean outcomes.'),
 html.Label('Launch site'),dcc.Dropdown(id='site',options=[{'label':'All sites','value':'ALL'}]+[{'label':s,'value':s} for s in sorted(df.LaunchSite.unique())],value='ALL',clearable=False),
 html.Label('Payload mass (kg)'),dcc.RangeSlider(id='payload',min=0,max=16000,step=500,value=[0,16000],marks={0:'0',4000:'4,000',8000:'8,000',12000:'12,000',16000:'16,000'}),
 dcc.Graph(id='success-pie'),dcc.Graph(id='payload-scatter'),
 html.P('The API and SQL snapshots differ: the 2